# Skill extraction rules (spaCy EntityRuler)


TIER 1 - PHRASES (any case)  
        Long / distinctive names. "projektledning", "Projektledning" and "PROJEKTLEDNING" all match. This is the default tier.  
        Example: "Google Cloud Platform", "Bluetooth-teknik".
 
TIER 2 - EXACT CASE (short or ambiguous words)
    Short acronyms and words that are also ordinary words.
    Only the exact spelling matches. "SAP" matches, "sap" does not.  
    Example: "R", "SAP", "REST", "Node", "Dart".
 
TIER 3 - CUSTOM ALIASES (synonyms -> one canonical skill)
    Different spellings / languages / abbreviations that should all map to ONE canonical name. Each alias is declared as any-case or exact-case, because "Machine learning" is safe in any case but "ML" and "AI" are not.


In [1]:
import re
import spacy

## Specific for me
**Allmänna datakunskaper**:  
    Systemanalys, Systemarkitektur, Verksamhetsutveckling, Systemintegration, Cybersäkerhet   

**Applikationsplattformar**:  
    Kubernetes, Red Hat OpenShift, Oracle Cloud Infrastructure (OCI), Oracle Service Bus, Oracle SOA Suite, MuleSoft Anypoint Platform 

**Certifikat/licenser**:    
    Certified Business Analysis Professional (CBAP), Entry Certificate in Business Analysis (ECBA), TOGAF-certifiering, AWS Certified Solutions Architect, Microsoft Certified: Azure Solutions Architect Expert

In [2]:
# TIER 1 - PHRASES, matched in ANY case
# Taxonomy labels, copied as-is. Labels with "/" or "(...)" are expanded into aliases automatically.

TIER1_PHRASES = [
"Kravhantering", 
"Systemanalys", # New
"Systemarkitektur", # New
"Systemintegration", # New
"Verksamhetsutveckling", # New
"Cybersäkerhet", # New
"Oracle SOA Suite", # New 
"Docker",
"IBM Cloud", 
"Sun Java Enterprise System",
"Windows Server", 
"Associate Android Developer",
"Oracle Certified Associate",
"PHP Developer Certification",
"AWS Certified Solutions Architect" # New
]


# TIER 2 - EXACT CASE, short or ambiguous words
# {canonical skill: [exact spellings]}

TIER2_EXACT_CASE = {
    "A+": ["A+"],
    "Odoo": ["Odoo"],
    "SAP": ["SAP"],
}


# TIER 3 - CUSTOM ALIASES
# {canonical skill: {"any_case": [...], "exact_case": [...]}}

TIER3_CUSTOM = {
    "Internet Information Services": {
        "any_case": ["Internet Information Services", "Internet Information Services (IIS)"],
        "exact_case": ["IIS"],
    },
    "Oracle Cloud Infrastructure": {
        "any_case": ["Oracle Cloud Infrastructure", "Oracle Cloud Infrastructure (OCI)", "Oracle Cloud"],
        "exact_case": ["OCI"],
    }, # New
    "Amazon Web Services": {
        "any_case": ["Amazon Web Services", "Amazon Web Services (AWS)"],
        "exact_case": ["AWS"],
    },
    "Infor": {
        "any_case": ["Infor OS", "Infor CloudSuite"],
        "exact_case": ["Infor"],
    },
    "Microsoft Dynamics 365": {
      "any_case": ["Microsoft Dynamics 365", "Dynamics 365"],
      "exact_case": ["D365"],
  },
    "Office SharePoint Server": {
        "any_case": ["Office SharePoint Server", "Office SharePoint Server (MOSS)", "SharePoint Server", "SharePoint", "SharePoint Online"],
        "exact_case": ["MOSS"],
    },
    "Oracle JCAPS": {
        "any_case": ["Oracle Java CAPS", "Java Composite Application Platform Suite"],
        "exact_case": ["Java CAPS"],
    }, # this was discontinued in 2016
    "Oracle Service Bus": {
        "any_case": ["Oracle Service Bus", "Oracle Service Bus (OSB)"],
        "exact_case": ["OSB"],
    }, # New
    "Oracle NetSuite": {
        "any_case": ["Oracle NetSuite", "NetSuite"],
    },
    "Salesforce Cloud": {
        "any_case": ["Salesforce Cloud", "Service Cloud", "Sales Cloud", "Salesforce"],
    },
    "MuleSoft Anypoint Platform": {
        "any_case": ["MuleSoft Anypoint Platform", "Mulesoft-plattformen", "MuleSoft"],
    }, # New
    "Red Hat OpenShift": {
        "any_case": ["Red Hat OpenShift", "OpenShift"],
    }, # New
    "AZ-204: Developing Solutions for Microsoft Azure": {
        "any_case": ["AZ-204: Developing Solutions for Microsoft Azure", "Developing Solutions for Microsoft Azure"],
        "exact_case": ["AZ-204"],
    },
    "AZ-900: Microsoft Azure Fundamentals": {
        "any_case": ["AZ-900: Microsoft Azure Fundamentals", "Microsoft Azure Fundamentals"],
        "exact_case": ["AZ-900"],
    },
    "MB-600: Microsoft Dynamics 365 Power Platform Solution Architect": {
        "any_case": ["MB-600: Microsoft Dynamics 365 Power Platform Solution Architect", "Dynamics 365 + Power Platform Solution Architect Expert"],
        "exact_case": ["MB-600"],
    },
    "Microsoft 365 Certified: Fundamentals": {
        "any_case": ["Microsoft 365 Certified: Fundamentals", "Microsoft Certified: Fundamentals"],
        "exact_case": ["MS-900"],
    },
    "Microsoft Certified: Application Developer": {
        "any_case": ["Microsoft Certified: Application Developer", "Microsoft Certified: Application Developer (MCAD)"],
        "exact_case": ["MCAD"],
    },
    "Microsoft Certified: Database Administrator": {
        "any_case": ["Microsoft Certified: Database Administrator", "Microsoft Certified: Database Administrator (MCDBA)"],
        "exact_case": ["MCDBA"],
    },
    "Microsoft Certified: Professional": {
        "any_case": ["Microsoft Certified: Professional", "Microsoft Certified: Professional (MCP)"],
    },
    "Microsoft Certified: Solution Developer": {
        "any_case": ["Microsoft Certified: Solution Developer", "Microsoft Certified: Solution Developer (MCSD)"],
        "exact_case": ["MCSD"],
    },
    "Microsoft Certified: System Engineer": {
        "any_case": ["Microsoft Certified: System Engineer", "Microsoft Certified: System Engineer (MCSE)"],
        "exact_case": ["MCSE"],
    },
    "Microsoft Certified: Azure Administrator Associate": {
        "any_case": ["Microsoft Certified: Azure Administrator Associate"],
    },
    "Microsoft Certified: Azure Solutions Architect Expert": {
        "any_case": ["Microsoft Certified: Azure Solutions Architect Expert", "Azure Solutions Architect Expert"],
    }, # New
    "Oracle Certified Database Administrator": {
        "any_case": ["Oracle Certified Database Administrator", "Oracle Certified Database Administrator (OCDBA)"],
        "exact_case": ["OCDBA"],
    },
    "Oracle Certified Database Operator": {
        "any_case": ["Oracle Certified Database Operator", "Oracle Certified Database Operator (DBO)"],
        "exact_case": ["DBO"],
    },
    "Oracle Certified Professional": {
        "any_case": ["Oracle Certified Professional", "Oracle Certified Professional/OCP", "Oracle Certified Professional (OCP)"],
    },
    "Certified Business Analysis Professional": {
        "any_case": ["Certified Business Analysis Professional", "Certified Business Analysis Professional (CBAP)"],
        "exact_case": ["CBAP"],
    }, # New
    "Entry Certificate in Business Analysis": {
        "any_case": ["Entry Certificate in Business Analysis", "Entry Certificate in Business Analysis (ECBA)"],
        "exact_case": ["ECBA"],
    }, # New
    "TOGAF-certifiering": {
        "any_case": ["TOGAF-certifiering"],
        "exact_case": ["TOGAF"],
    }, # New
    "Certified JavaScript Developer": {
        "any_case": ["Certified JavaScript Developer", "JavaScript Developer-certifiering"],
    }, # New
    "Windows Azure": {
        "any_case": ["Microsoft Azure", "Azure"],
    },
    "Google Cloud Platform": {
        "any_case": ["Google Cloud"],
        "exact_case": ["GCP"]
    },
    "Kubernetes": {
        "any_case": ["Kubernetes"],
        "exact_case": ["K8s"]
    }, # New
    "Apache Kafka": {
        "any_case": ["Apache Kafka", "Kafka"],
    },
    "Apache Tomcat": {
        "any_case": ["Apache Tomcat", "Tomcat"],
    },
    "Weblogic Server": {
        "any_case": ["Weblogic Server", "WebLogic", "Oracle WebLogic"],
    },
    "JBoss Application Server": {
        "any_case": ["JBoss", "JBoss EAP"],
    },
    "IBM WebSphere": {
        "any_case": ["IBM WebSphere", "WebSphere"],
    },
    "Glassfish Server": {
        "any_case": ["Glassfish Server", "GlassFish"],
    },
    "IBM Domino": {
        "any_case": ["IBM Domino", "HCL Domino", "Lotus Domino"],
    },
    "Windows Azure": {
        "any_case": ["Microsoft Azure", "Azure"],
    },
    "Windows Azure": {
        "any_case": ["Microsoft Azure", "Azure"],
    },
    "Windows Azure": {
        "any_case": ["Microsoft Azure", "Azure"],
    },
    "Windows Azure": {
        "any_case": ["Microsoft Azure", "Azure"],
    },
    "Windows Azure": {
        "any_case": ["Microsoft Azure", "Azure"],
    },
    "Windows Azure": {
        "any_case": ["Microsoft Azure", "Azure"],
    },
    "Windows Azure": {
        "any_case": ["Microsoft Azure", "Azure"],
    },
    "Windows Azure": {
        "any_case": ["Microsoft Azure", "Azure"],
    },
    "Windows Azure": {
        "any_case": ["Microsoft Azure", "Azure"],
    },
    "Windows Azure": {
        "any_case": ["Microsoft Azure", "Azure"],
    },

 } 

## Ignore the following, it is just to test

In [3]:
# Compile the three tiers into two lookups: {form: canonical}
any_case, exact_case = {}, {}

for label in TIER1_PHRASES:
    parts = [p.strip() for p in re.split(r"[/()]", label) if p.strip()]
    for p in parts:
        if len(parts) > 1 and p.isupper():   # abbreviation such as AWS
            exact_case[p] = label
        else:
            any_case[p] = label

for canonical, forms in TIER2_EXACT_CASE.items():
    for f in forms:
        exact_case[f] = canonical

for canonical, groups in TIER3_CUSTOM.items():
    for f in groups.get("any_case", []):
        any_case[f] = canonical
    for f in groups.get("exact_case", []):
        exact_case[f] = canonical

# Two rulers: any-case (LOWER) first, exact-case (literal text) second
nlp = spacy.blank("sv")

def add_ruler(name, rules, lower):
    config = {"phrase_matcher_attr": "LOWER"} if lower else {}
    ruler = nlp.add_pipe("entity_ruler", name=name, config=config)
    ruler.add_patterns(
        [{"label": "SKILL", "pattern": form, "id": canon} for form, canon in rules.items()]
    )

add_ruler("any_case", any_case, lower=True)
add_ruler("exact_case", exact_case, lower=False)

### Tests

In [4]:
# 1. Systemanalytiker till Region Skåne-liknande förvaltning

text_1 = "Vi söker en systemanalytiker som vill driva verksamhetsutveckling tillsammans med våra verksamhetsexperter. Du ansvarar för kravhantering och systemanalys i flera parallella projekt och översätter behov till tydliga specifikationer. Vi ser gärna att du har en certifiering som CBAP eller ECBA, och erfarenhet av ramverket TOGAF. Vår intranätplattform bygger på Office SharePoint Server, så vana vid den miljön är meriterande. Du arbetar nära arkitekter och utvecklare och bidrar till vår systemarkitektur."

for ent in nlp(text_1).ents:
    print(f"{ent.text:<7} -> {ent.ent_id_}")

verksamhetsutveckling -> Verksamhetsutveckling
kravhantering -> Kravhantering
systemanalys -> Systemanalys
CBAP    -> Certified Business Analysis Professional
ECBA    -> Entry Certificate in Business Analysis
TOGAF   -> TOGAF-certifiering
Office SharePoint Server -> Office SharePoint Server
systemarkitektur -> Systemarkitektur


In [5]:
# 2. IT-arkitekt, bank och finans

text_2 = "Som IT-arkitekt hos oss tar du ansvar för systemarkitektur och systemintegration i en komplex miljö. " \
"Vi integrerar med hjälp av Oracle SOA Suite, Oracle Service Bus (OSB) och MuleSoft Anypoint Platform, och strömmar händelser via Apache Kafka. " \
"Vår nya plattform körs på Red Hat OpenShift, så god förståelse för Kubernetes och Docker är ett krav. Vi har delar av verksamheten i Oracle Cloud Infrastructure. " \
"Cybersäkerhet är centralt i allt vi gör. En TOGAF-certifiering är ett plus."

for ent in nlp(text_2).ents:
    print(f"{ent.text:<7} -> {ent.ent_id_}")

systemarkitektur -> Systemarkitektur
systemintegration -> Systemintegration
Oracle SOA Suite -> Oracle SOA Suite
Oracle Service Bus (OSB) -> Oracle Service Bus
MuleSoft Anypoint Platform -> MuleSoft Anypoint Platform
Apache Kafka -> Apache Kafka
Red Hat OpenShift -> Red Hat OpenShift
Kubernetes -> Kubernetes
Docker  -> Docker
Oracle Cloud Infrastructure -> Oracle Cloud Infrastructure
Cybersäkerhet -> Cybersäkerhet
TOGAF-certifiering -> TOGAF-certifiering


In [6]:
# 3. Molnutvecklare (Azure, AWS, GCP)

text_3 = "Vi bygger molnbaserade lösningar och behöver förstärka teamet. Du har arbetat med Amazon Web Services och gärna tagit AWS Certified Solutions Architect. " \
"Vi använder också Windows Azure och Google Cloud Platform, så certifieringar som AZ-204, AZ-900 eller Azure Solutions Architect Expert värderas högt. " \
"Du är van vid containers med Docker och att skriva kod i JavaScript, och en JavaScript Developer-certifiering är meriterande. " \
"Vi har även äldre webbapplikationer i PHP där en PHP Developer Certification kommer till nytta."

for ent in nlp(text_3).ents:
    print(f"{ent.text:<7} -> {ent.ent_id_}")

Amazon Web Services -> Amazon Web Services
AWS Certified Solutions Architect -> AWS Certified Solutions Architect
Azure   -> Windows Azure
Google Cloud -> Google Cloud Platform
AZ-204  -> AZ-204: Developing Solutions for Microsoft Azure
AZ-900  -> AZ-900: Microsoft Azure Fundamentals
Azure Solutions Architect Expert -> Microsoft Certified: Azure Solutions Architect Expert
Docker  -> Docker
JavaScript Developer-certifiering -> Certified JavaScript Developer
PHP Developer Certification -> PHP Developer Certification


In [7]:
# 4. ERP-konsult, affärssystem

text_4 = "Vi är ett konsultbolag som hjälper företag att välja och implementera affärssystem. " \
"Du bör ha erfarenhet av SAP och gärna av Infor OS eller Infor CloudSuite. " \
"Våra mindre kunder kör Odoo eller Oracle NetSuite, medan större kunder väljer Microsoft Dynamics 365, där certifieringen MB-600 är mycket meriterande. " \
"Kunskap om Salesforce Cloud, främst Sales Cloud och Service Cloud, är också bra att ha. Du leder kravhantering med kunden och bidrar till deras verksamhetsutveckling. " \
"Vi har kunder som köper certifierings- och utbildningstjänster, så vi gillar även MS-900."

for ent in nlp(text_4).ents:
    print(f"{ent.text:<7} -> {ent.ent_id_}")

SAP     -> SAP
Infor OS -> Infor
Infor CloudSuite -> Infor
Odoo    -> Odoo
Oracle NetSuite -> Oracle NetSuite
Microsoft Dynamics 365 -> Microsoft Dynamics 365
MB-600  -> MB-600: Microsoft Dynamics 365 Power Platform Solution Architect
Salesforce Cloud -> Salesforce Cloud
Sales Cloud -> Salesforce Cloud
Service Cloud -> Salesforce Cloud
kravhantering -> Kravhantering
verksamhetsutveckling -> Verksamhetsutveckling
MS-900  -> Microsoft 365 Certified: Fundamentals


In [8]:
# 5. Drift- och applikationsförvaltare, äldre miljöer

text_5 = "Vi söker dig som tar hand om vår driftmiljö, där många system fortfarande är traditionella. " \
"Du hanterar Windows Server med IIS, samt applikationsservrar som Apache Tomcat, Weblogic Server, IBM WebSphere, JBoss Application Server och Glassfish Server. " \
"Vi har också kvar IBM Domino, en äldre Sun Java Enterprise System-installation och ett par integrationer i Oracle Java CAPS. Delar av tjänsterna har flyttat till IBM Cloud. " \
"Certifieringar som A+, MCSE, MCP, MCSD, MCAD, MCDBA, OCP, OCDBA, DBO, Oracle Certified Associate, Associate Android Developer och Microsoft Certified: Azure Administrator Associate ses som meriter."
for ent in nlp(text_5).ents:
    print(f"{ent.text} -> {ent.ent_id_}")

Windows Server -> Windows Server
IIS -> Internet Information Services
Apache Tomcat -> Apache Tomcat
Weblogic Server -> Weblogic Server
IBM WebSphere -> IBM WebSphere
JBoss -> JBoss Application Server
Glassfish Server -> Glassfish Server
IBM Domino -> IBM Domino
Oracle Java CAPS -> Oracle JCAPS
IBM Cloud -> IBM Cloud
A+ -> A+
MCSE -> Microsoft Certified: System Engineer
MCSD -> Microsoft Certified: Solution Developer
MCAD -> Microsoft Certified: Application Developer
MCDBA -> Microsoft Certified: Database Administrator
OCDBA -> Oracle Certified Database Administrator
DBO -> Oracle Certified Database Operator
Oracle Certified Associate -> Oracle Certified Associate
Associate Android Developer -> Associate Android Developer
Microsoft Certified: Azure Administrator Associate -> Microsoft Certified: Azure Administrator Associate
